In [ ]:
%pip install ipykernel
%pip install opencv-python 
%pip install matplotlib
%pip install matplotlib tk
%pip install plotly nbformat
%pip install scipy
%pip install pyqtgraph PyOpenGL PyQt5
%matplotlib qt


In [ ]:
import time
from collections import deque
from dataclasses import dataclass

import cv2 as cv
import matplotlib
import numpy as np

In [ ]:
import threading
import time
from collections import deque
from dataclasses import dataclass

import cv2 as cv
import matplotlib
import numpy as np


def use_window_backend():
    """Notebooks default to an inline backend that closes live figures."""
    try:
        shell = get_ipython()
    except NameError:
        return
    for backend in ("qt", "tk"):
        try:
            shell.run_line_magic("matplotlib", backend)
            return
        except Exception:
            continue


use_window_backend()
import matplotlib.pyplot as plt  # noqa: E402
from matplotlib.cm import ScalarMappable  # noqa: E402
from matplotlib.colors import Normalize  # noqa: E402


FRAME_SHAPE = (384, 256, 2)
FRAME_SIZE = int(np.prod(FRAME_SHAPE))
FONT = cv.FONT_HERSHEY_SIMPLEX


@dataclass
class Config:
    camera_index: int = 0
    left_crop: int = 98              # gray (fallback) mode only
    zoom: int = 4
    rows: int = 80                   # time rows kept in the 3D views
    columns: int = 64                # target number of 3D columns across the image
    update_seconds: float = 0.08     # time between 3D updates
    figure_size: tuple = (18, 8)
    blur_x: float = 0.8              # smoothing across image columns (3D views)
    blur_t: float = 1.0              # smoothing across time rows (3D views)

    # temperature scale (raw mode). Starts at temp_min/max, then follows the scene.
    temp_min: float = 10.0
    temp_max: float = 50.0
    auto_scale: bool = True
    scale_low_pct: float = 1.0
    scale_high_pct: float = 99.8
    min_span: float = 6.0            # never zoom the scale tighter than this many degrees
    scale_smoothing: float = 0.08    # 0..1, lower = steadier scale

    # wisps
    ambient_pct: float = 30.0        # "normal" = this percentile of the recent history
    contrast: float = 0.15           # fraction of the scale span that makes a wisp fully strong
    min_strength: float = 0.15       # weaker cells are not drawn


class Scale:
    """Colour/height range shared by the camera view, the bar and both 3D views."""

    def __init__(self, low, high, unit, auto, cfg):
        self.low, self.high, self.unit, self.auto, self.cfg = low, high, unit, auto, cfg
        self.primed = False

    @property
    def span(self):
        return max(self.high - self.low, 1e-6)

    @property
    def label(self):
        return "°C" if self.unit == "C" else self.unit

    def update(self, values):
        """Follow the scene smoothly; rise fast so hot things never clip."""
        if not self.auto:
            return
        cfg = self.cfg
        lo, hi = np.percentile(values[::2, ::2], (cfg.scale_low_pct, cfg.scale_high_pct))
        hi = max(hi, lo + cfg.min_span)
        if not self.primed:
            self.low, self.high, self.primed = float(lo), float(hi), True
            return
        self.low += cfg.scale_smoothing * (lo - self.low)
        self.high += (0.5 if hi > self.high else cfg.scale_smoothing) * (hi - self.high)


def decode_temps(frame):
    """Return a 192x256 array in °C from a raw TC001 frame."""
    data = frame.reshape(FRAME_SHAPE)[FRAME_SHAPE[0] // 2:]
    raw = data[..., 0].astype(np.uint16) + data[..., 1].astype(np.uint16) * 256
    return (raw / 64.0 - 273.15).astype(np.float32)


def decode_gray(frame, crop):
    """Return the cropped brightness image (0 to 255) from a normal colour frame."""
    return cv.cvtColor(frame[:, crop:], cv.COLOR_BGR2GRAY).astype(np.float32)


def open_raw(index):
    """Return a capture that delivers raw TC001 frames, or None if it doesn't."""
    for name, backend in (("DSHOW", cv.CAP_DSHOW), ("MSMF", cv.CAP_MSMF), ("default", cv.CAP_ANY)):
        capture = cv.VideoCapture(index, backend)
        if not capture.isOpened():
            capture.release()
            continue
        capture.set(cv.CAP_PROP_CONVERT_RGB, 0)
        ok, frame = capture.read()
        print(f"[{name}] read ok={ok}, shape={getattr(frame, 'shape', None)}, "
              f"size={getattr(frame, 'size', None)} (need {FRAME_SIZE})")
        if ok and frame.size == FRAME_SIZE:
            return capture
        capture.release()
    return None


class Camera:
    """Reads frames on a background thread so the display never works from stale, queued frames."""

    def __init__(self, cfg):
        self.crop = cfg.left_crop
        self.capture = open_raw(cfg.camera_index)
        self.raw = self.capture is not None
        if not self.raw:
            self.capture = cv.VideoCapture(cfg.camera_index)
        if not self.capture.isOpened():
            raise RuntimeError("Could not open camera. Try another camera_index.")
        if self.raw:
            self.scale = Scale(cfg.temp_min, cfg.temp_max, "C", cfg.auto_scale, cfg)
        else:
            self.scale = Scale(0, 255, "signal", False, cfg)

        self._frame = None
        self._lock = threading.Lock()
        self._ready = threading.Event()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._grab, daemon=True)
        self._thread.start()

    def _grab(self):
        while not self._stop.is_set():
            ok, frame = self.capture.read()
            with self._lock:
                self._frame = frame if ok else None
            self._ready.set()
            if not ok:
                break

    def read(self):
        """Return the newest temperatures in °C (raw mode) or brightness, or None."""
        if not self._ready.wait(2.0):
            return None
        self._ready.clear()
        with self._lock:
            frame = self._frame
        if frame is None:
            return None
        return decode_temps(frame) if self.raw else decode_gray(frame, self.crop)

    def release(self):
        self._stop.set()
        self._thread.join(timeout=2.0)
        self.capture.release()


class CameraView:
    """Camera window with a hotspot marker and a temperature bar."""

    def __init__(self, cfg, scale):
        self.zoom = cfg.zoom
        self.scale = scale
        self.bar = None

    def make_bar(self, height):
        gradient = np.linspace(255, 0, height).astype(np.uint8)[:, None].repeat(22, axis=1)
        return cv.applyColorMap(gradient, cv.COLORMAP_INFERNO)

    def show(self, values):
        s = self.scale
        scaled = np.clip((values - s.low) / s.span, 0, 1)
        image = cv.applyColorMap(np.uint8(scaled * 255), cv.COLORMAP_INFERNO)
        image = cv.resize(image, None, fx=self.zoom, fy=self.zoom, interpolation=cv.INTER_CUBIC)
        h, w = image.shape[:2]

        y, x = np.unravel_index(np.argmax(values), values.shape)
        cx, cy = int((x + 0.5) * self.zoom), int((y + 0.5) * self.zoom)
        cv.drawMarker(image, (cx, cy), (255, 255, 255), cv.MARKER_CROSS, 14, 1)
        cv.putText(image, f"{values[y, x]:.1f} {s.unit}", (min(cx + 8, w - 120), max(cy - 8, 16)),
                   FONT, 0.5, (255, 255, 255), 1, cv.LINE_AA)

        if self.bar is None or self.bar.shape[0] != h:
            self.bar = self.make_bar(h)
        labels = np.zeros((h, 80, 3), np.uint8)
        for i in range(6):
            frac = i / 5
            row = int(frac * (h - 1))
            cv.line(labels, (0, row), (5, row), (255, 255, 255), 1)
            cv.putText(labels, f"{s.high - frac * s.span:.1f}", (9, min(max(row + 5, 12), h - 3)),
                       FONT, 0.45, (255, 255, 255), 1, cv.LINE_AA)
        spacer = np.zeros((h, 8, 3), np.uint8)
        cv.imshow("TC001", np.hstack([image, spacer, self.bar, labels]))

    @staticmethod
    def quit_requested():
        return cv.waitKey(1) & 0xFF in (ord("q"), 27)


class History:
    """Rolling rows: hottest value in each block of image columns, averaged over each time step."""

    def __init__(self, width, cfg):
        self.step = max(1, width // cfg.columns)
        self.n = width // self.step
        self.x = np.arange(self.n) * self.step + self.step / 2
        self.rows = deque(maxlen=cfg.rows)
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def add_frame(self, values):
        blurred = cv.GaussianBlur(values, (0, 0), 1.0)        # suppress single hot pixels
        h = blurred.shape[0]
        blocks = blurred[:, :self.n * self.step].reshape(h, self.n, self.step)
        self.total += blocks.max(axis=(0, 2))                  # hottest pixel per column block
        self.count += 1

    def commit(self):
        if self.count == 0:
            return
        row = self.total / self.count
        if not self.rows:
            self.rows.extend([row] * self.rows.maxlen)
        else:
            self.rows.append(row)
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def as_array(self):
        return np.array(self.rows, np.float32)


def style_dark_axes(ax, xlim, ylim, zlabel):
    ax.set_facecolor("black")
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.set_pane_color((0, 0, 0, 0))
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_xlabel("Image X", color="0.7")
    ax.set_ylabel("Time", color="0.7")
    ax.set_zlabel(zlabel, color="0.7")
    ax.tick_params(colors="0.6")
    ax.view_init(45, -70)


class SurfacePanel:
    """Left panel: surface of the hottest value per column over time, in real units."""

    def __init__(self, ax, history, cfg, scale):
        self.ax, self.scale, self.surf = ax, scale, None
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))
        style_dark_axes(ax, (0, history.x.max()), (0, cfg.rows - 1), f"Signal ({scale.label})")
        ax.set_title("3D map", color="0.85")

    def draw(self, grid):
        s = self.scale
        if self.surf is not None:
            self.surf.remove()
        self.surf = self.ax.plot_surface(
            self.x, self.y, np.clip(grid, s.low, s.high),
            cmap="inferno", vmin=s.low, vmax=s.high,
            linewidth=0, antialiased=False, shade=False,
            rcount=grid.shape[0], ccount=grid.shape[1],
        )
        self.ax.set_zlim(s.low, s.high)


class WispPanel:
    """Right panel: particles for everything hotter than the scene's normal level."""

    PARTICLES = 2

    def __init__(self, ax, history, cfg, scale):
        self.ax, self.cfg, self.scale, self.history = ax, cfg, scale, history
        self.x, self.y = np.meshgrid(history.x, np.arange(cfg.rows))
        rng = np.random.default_rng(7)
        self.jitter = rng.uniform(-0.5, 0.5, (self.PARTICLES, 3, self.x.size))   # fixed, so no flicker
        self.fade = np.linspace(0.15, 1.0, cfg.rows)[:, None]
        self.ambient = None
        self.top = None
        style_dark_axes(ax, (0, history.x.max()), (0, cfg.rows - 1), f"{scale.label} above normal")
        ax.set_title("Wisps", color="0.85")
        self.cloud = ax.scatter([0], [0], [0], s=1, c=[(0, 0, 0, 0)], depthshade=False, linewidths=0)

    def draw(self, grid):
        s, cfg = self.scale, self.cfg
        target = float(np.percentile(grid, cfg.ambient_pct))
        self.ambient = target if self.ambient is None else self.ambient + 0.05 * (target - self.ambient)

        contrast = max(cfg.contrast * s.span, 1e-6)
        excess = np.clip(grid - self.ambient, 0, None)
        strength = np.clip(excess / contrast, 0, 1) * self.fade
        cells = np.flatnonzero(strength.ravel() > cfg.min_strength)

        target_top = max(4 * contrast, float(excess.max()) * 1.1)
        self.top = target_top if self.top is None else self.top + 0.2 * (target_top - self.top)
        self.ax.set_zlim(0, self.top)

        if not cells.size:
            self.cloud._offsets3d = ([0.0], [0.0], [0.0])
            self.cloud.set_facecolor([(0, 0, 0, 0)])
            self.cloud.set_sizes([1])
            return

        heat = excess.ravel()[cells]
        power = strength.ravel()[cells]
        jit = self.jitter[:, :, cells]                                        # (P, 3, n)
        px = (self.x.ravel()[cells] + jit[:, 0] * self.history.step).ravel()
        py = (self.y.ravel()[cells] + jit[:, 1]).ravel()
        pz = np.clip(heat + jit[:, 2] * contrast * 0.15, 0, None).ravel()
        power = np.tile(power, self.PARTICLES)
        temp = self.ambient + np.tile(heat, self.PARTICLES)                    # absolute value -> same colour bar

        colors = plt.get_cmap("inferno")(np.clip((temp - s.low) / s.span, 0, 1))
        colors[:, 3] = np.clip(power * 1.2, 0, 1)
        self.cloud._offsets3d = (px, py, pz)
        self.cloud.set_facecolor(colors)
        self.cloud.set_sizes(4 + 34 * power)


class SideBySide:
    """One window: surface map (left), wisp map (right), shared temperature bar."""

    def __init__(self, history, cfg, scale):
        plt.ion()
        self.history, self.cfg, self.scale = history, cfg, scale
        self.fig = plt.figure("Thermal 3D + wisps", figsize=cfg.figure_size, facecolor="black")
        left = self.fig.add_subplot(121, projection="3d")
        right = self.fig.add_subplot(122, projection="3d")
        self.fig.subplots_adjust(left=0.01, right=0.9, bottom=0.02, top=0.95, wspace=0.02)

        self.mappable = ScalarMappable(norm=Normalize(scale.low, scale.high), cmap="inferno")
        self.cbar = self.fig.colorbar(self.mappable, cax=self.fig.add_axes([0.93, 0.2, 0.014, 0.6]))
        self.cbar.set_label(f"Temperature ({scale.label})", color="0.8")
        self.cbar.ax.tick_params(colors="0.8")
        self.cbar.outline.set_edgecolor("0.5")

        self.panels = [
            SurfacePanel(left, history, cfg, scale),
            WispPanel(right, history, cfg, scale),
        ]

    def draw(self):
        """Redraw both panels. Returns False once the window is closed."""
        grid = cv.GaussianBlur(self.history.as_array(), (0, 0),
                               sigmaX=self.cfg.blur_x, sigmaY=self.cfg.blur_t)
        self.mappable.set_clim(self.scale.low, self.scale.high)
        self.cbar.update_normal(self.mappable)
        for panel in self.panels:
            panel.draw(grid)
        plt.pause(0.001)
        return plt.fignum_exists(self.fig.number)


def run(cfg=Config()):
    if "inline" in matplotlib.get_backend().lower():
        raise RuntimeError("Inline backend closes the figure. Run %matplotlib qt, then restart the kernel.")

    camera = Camera(cfg)
    print("Reading temperatures (°C)." if camera.raw else "Reading brightness only: this camera sends processed frames.")
    view = CameraView(cfg, camera.scale)
    history = maps = None
    last_update = 0.0

    try:
        while True:
            values = camera.read()
            if values is None:
                break
            camera.scale.update(values)

            if history is None:
                history = History(values.shape[1], cfg)
                maps = SideBySide(history, cfg, camera.scale)

            view.show(values)
            history.add_frame(values)

            if time.monotonic() - last_update >= cfg.update_seconds:
                last_update = time.monotonic()
                history.commit()
                if not maps.draw():
                    break

            if view.quit_requested():
                break
    finally:
        camera.release()
        cv.destroyAllWindows()
        plt.close("all")


if __name__ == "__main__":
    run()
    

In [ ]:
#simulated - one window: live map with hotspot boxes, tracked-only view, 3D views and data panels
# Drawn with pyqtgraph + OpenGL (not matplotlib), so every part updates every frame without stalls.
import threading
import time
from collections import deque
from dataclasses import dataclass
from itertools import takewhile

import cv2 as cv
import numpy as np
import pyqtgraph as pg
import pyqtgraph.opengl as gl
from pyqtgraph.Qt import QtCore, QtGui, QtWidgets

QShortcut = getattr(QtGui, "QShortcut", None) or QtWidgets.QShortcut   # Qt6 / Qt5


FRAME_SHAPE = (384, 256, 2)
FRAME_SIZE = int(np.prod(FRAME_SHAPE))


def wait_until(deadline, idle):
    """Sleep until `deadline`, keeping the window responsive while waiting."""
    while True:
        left = deadline - time.monotonic()
        if left <= 0:
            return
        if idle is not None:
            idle()
        time.sleep(min(left, 0.003))


@dataclass
class Config:
    simulate: bool = True            # True = synthetic scene, False = real TC001
    sim_fps: float = 25.0            # simulated frame rate
    camera_index: int = 0
    left_crop: int = 98              # gray (fallback) mode only
    rows: int = 80                   # time rows kept in the 3D views
    columns: int = 64                # target number of 3D columns across the image
    update_seconds: float = 0.08     # time between new rows in the 3D views
    full_update_seconds: float = 0.2 # time between list / data panel / chart refreshes
    figure_size: tuple = (18, 10)
    blur_x: float = 0.8              # smoothing across image columns (3D views)
    blur_t: float = 1.0              # smoothing across time rows (3D views)

    # temperature scale (raw mode). Starts at temp_min/max, then follows the scene.
    temp_min: float = 10.0
    temp_max: float = 50.0
    auto_scale: bool = True
    scale_low_pct: float = 1.0
    scale_high_pct: float = 99.8
    min_span: float = 6.0            # never zoom the scale tighter than this many degrees
    scale_smoothing: float = 0.08    # 0..1, lower = steadier scale

    # wisps
    ambient_pct: float = 30.0        # "normal" = this percentile of the recent history
    contrast: float = 0.15           # fraction of the scale span that makes a wisp fully strong
    min_strength: float = 0.15       # weaker cells are not drawn

    # hotspot detection and tracking
    hot_delta: float = 4.0           # a hotspot is at least this much hotter than the scene median...
    hot_fraction: float = 0.12       # ...or this fraction of the colour scale, whichever is larger
    prominence: float = 5.0          # a peak must rise this much above the dip joining it to a
                                     # hotter neighbour to count as its own hotspot
    min_area: int = 20               # pixels; smaller hot blobs are ignored
    min_rise: float = 1.5            # a hotspot's peak must be this far above the hotspot level
    detect_step: int = 2             # detection runs on an image this many times smaller (speed)
    match_distance: float = 30.0     # pixels a hotspot may jump between frames and keep its ID...
    max_speed: float = 80.0          # ...plus this many pixels per second since it was last seen
    confirm_frames: int = 3          # frames before a new hotspot gets an ID and a box
    forget_frames: int = 20          # frames a hotspot may be missing before it is dropped
    focus_margin: int = 4            # pixels of surroundings kept around the picked hotspot
    chart_seconds: float = 30.0      # length of the temperature history chart
    log_length: int = 30000          # samples kept per hotspot (for the chart and CSV export)


class Scale:
    """Colour/height range shared by the camera views, the bar and both 3D views."""

    def __init__(self, low, high, unit, auto, cfg):
        self.low, self.high, self.unit, self.auto, self.cfg = low, high, unit, auto, cfg
        self.primed = False

    @property
    def span(self):
        return max(self.high - self.low, 1e-6)

    @property
    def label(self):
        return "°C" if self.unit == "C" else self.unit

    def update(self, values):
        """Follow the scene smoothly; rise fast so hot things never clip."""
        if not self.auto:
            return
        cfg = self.cfg
        lo, hi = np.percentile(values[::2, ::2], (cfg.scale_low_pct, cfg.scale_high_pct))
        hi = max(hi, lo + cfg.min_span)
        if not self.primed:
            self.low, self.high, self.primed = float(lo), float(hi), True
            return
        self.low += cfg.scale_smoothing * (lo - self.low)
        self.high += (0.5 if hi > self.high else cfg.scale_smoothing) * (hi - self.high)


def decode_temps(frame):
    """Return a 192x256 array in °C from a raw TC001 frame."""
    data = frame.reshape(FRAME_SHAPE)[FRAME_SHAPE[0] // 2:]
    raw = data[..., 0].astype(np.uint16) + data[..., 1].astype(np.uint16) * 256
    return (raw / 64.0 - 273.15).astype(np.float32)


def decode_gray(frame, crop):
    """Return the cropped brightness image (0 to 255) from a normal colour frame."""
    return cv.cvtColor(frame[:, crop:], cv.COLOR_BGR2GRAY).astype(np.float32)


def open_raw(index):
    """Return a capture that delivers raw TC001 frames, or None if it doesn't."""
    for name, backend in (("DSHOW", cv.CAP_DSHOW), ("MSMF", cv.CAP_MSMF), ("default", cv.CAP_ANY)):
        capture = cv.VideoCapture(index, backend)
        if not capture.isOpened():
            capture.release()
            continue
        capture.set(cv.CAP_PROP_CONVERT_RGB, 0)
        ok, frame = capture.read()
        print(f"[{name}] read ok={ok}, shape={getattr(frame, 'shape', None)}, "
              f"size={getattr(frame, 'size', None)} (need {FRAME_SIZE})")
        if ok and frame.size == FRAME_SIZE:
            return capture
        capture.release()
    return None


class Camera:
    """Reads frames on a background thread so the display never works from stale, queued frames."""

    def __init__(self, cfg):
        self.crop = cfg.left_crop
        self.capture = open_raw(cfg.camera_index)
        self.raw = self.capture is not None
        if not self.raw:
            self.capture = cv.VideoCapture(cfg.camera_index)
        if not self.capture.isOpened():
            raise RuntimeError("Could not open camera. Try another camera_index.")
        if self.raw:
            self.scale = Scale(cfg.temp_min, cfg.temp_max, "C", cfg.auto_scale, cfg)
        else:
            self.scale = Scale(0, 255, "signal", False, cfg)

        self._frame = None
        self._lock = threading.Lock()
        self._ready = threading.Event()
        self._stop = threading.Event()
        self._thread = threading.Thread(target=self._grab, daemon=True)
        self._thread.start()

    def _grab(self):
        while not self._stop.is_set():
            ok, frame = self.capture.read()
            with self._lock:
                self._frame = frame if ok else None
            self._ready.set()
            if not ok:
                break

    def read(self, idle=None):
        """Return the newest temperatures in °C (raw mode) or brightness, or None."""
        deadline = time.monotonic() + 2.0
        while not self._ready.wait(0.003 if idle else 2.0):
            if idle is None or time.monotonic() > deadline:
                return None
            idle()
        self._ready.clear()
        with self._lock:
            frame = self._frame
        if frame is None:
            return None
        return decode_temps(frame) if self.raw else decode_gray(frame, self.crop)

    def release(self):
        self._stop.set()
        self._thread.join(timeout=2.0)
        self.capture.release()


class SimulatedCamera:
    """Drop-in stand-in for Camera: same attributes and methods, synthetic 192x256 °C frames.

    Scene: a cool room with a warm right-hand wall and slow drifting texture, a person
    (body + head) walking side to side, a pulsing hot spot (like a soldering iron or mug),
    and a warm hand that sweeps across now and then. Plus a little sensor noise.
    """

    H, W = 192, 256

    def __init__(self, cfg):
        self.raw = True
        self.crop = cfg.left_crop
        self.scale = Scale(cfg.temp_min, cfg.temp_max, "C", cfg.auto_scale, cfg)
        self.period = 1.0 / cfg.sim_fps
        self.rng = np.random.default_rng(1)
        self.gx = np.arange(self.W, dtype=np.float32)
        self.gy = np.arange(self.H, dtype=np.float32)

        # Static, smooth room texture (about +/-1 degree) and a warm wall on the right.
        coarse = self.rng.normal(size=(12, 16)).astype(np.float32)
        texture = cv.resize(coarse, (self.W, self.H), interpolation=cv.INTER_CUBIC)
        wall = 3.0 * np.clip((self.gx - 150) / 106, 0, 1) ** 2
        self.room = 21.0 + 1.0 * texture + wall[None, :]

        self.t0 = time.monotonic()
        self.next_frame = self.t0

    def _blob(self, cx, cy, sx, sy, amp):
        bx = np.exp(-((self.gx - cx) ** 2) / (2 * sx * sx))
        by = np.exp(-((self.gy - cy) ** 2) / (2 * sy * sy))
        return amp * by[:, None] * bx[None, :]

    def read(self, idle=None):
        """Wait until the next frame is due (keeping the window alive), then return it (°C)."""
        wait_until(self.next_frame, idle)
        self.next_frame = max(self.next_frame + self.period, time.monotonic() - self.period)
        t = time.monotonic() - self.t0

        frame = self.room.copy()

        # Person walking back and forth, with a bit of bobbing.
        px = self.W / 2 + 70 * np.sin(0.35 * t)
        py = self.H * 0.62 + 6 * np.sin(1.1 * t)
        frame += self._blob(px, py, 22, 48, 11.0)                 # torso
        frame += self._blob(px, py - 58, 10, 12, 13.0)            # head

        # Pulsing hot spot, just beyond where the person turns around.
        frame += self._blob(238, 150, 5, 5, 24 + 8 * np.sin(1.7 * t))

        # A warm hand sweeping across the top every ~9 s.
        phase = (t % 9.0) / 9.0
        frame += self._blob(-20 + phase * (self.W + 40), 50, 8, 8, 14 * np.sin(np.pi * phase))

        frame += self.rng.normal(0, 0.08, frame.shape)
        return frame.astype(np.float32)

    def release(self):
        pass


def make_camera(cfg):
    return SimulatedCamera(cfg) if cfg.simulate else Camera(cfg)


# ----------------------------------------------------------------------------------------------
# Hotspot detection and tracking
# ----------------------------------------------------------------------------------------------

@dataclass
class Detection:
    box: tuple          # x, y, w, h in image pixels
    center: tuple       # heat-weighted centre (x, y)
    peak: float         # hottest reading inside the hotspot
    peak_xy: tuple      # where that reading is
    mean: float         # average reading over the hotspot
    area: int           # pixels
    mask: np.ndarray    # bool image, True inside the hotspot


def segment_peaks(img, threshold, prominence):
    """Label the pixels above threshold, one label per peak that clearly stands on its own.

    Pixels are visited hottest first. Each one joins the region of a neighbour it touches.
    When two regions meet, the cooler one is folded into the hotter one unless its peak rises
    at least `prominence` above the meeting point. So a head on a body stays one hotspot,
    while a person standing next to a hot mug gives two.
    """
    h, w = img.shape
    pw = w + 2
    padded = np.full((h + 2, pw), -np.inf, np.float32)
    padded[1:-1, 1:-1] = img
    flat = padded.ravel()
    idx = np.flatnonzero(flat > threshold)
    if idx.size == 0:
        return np.zeros((h, w), np.int32)

    order = idx[np.argsort(-flat[idx], kind="stable")]
    offsets = (-pw - 1, -pw, -pw + 1, -1, 1, pw - 1, pw, pw + 1)
    owner, parent, peak = {}, [], []

    def find(c):
        while parent[c] != c:
            parent[c] = parent[parent[c]]
            c = parent[c]
        return c

    for p, v in zip(order.tolist(), flat[order].tolist()):
        roots = []
        for o in offsets:
            c = owner.get(p + o)
            if c is not None:
                r = find(c)
                if r not in roots:
                    roots.append(r)
        if not roots:
            owner[p] = len(parent)
            parent.append(len(parent))
            peak.append(v)
            continue
        if len(roots) > 1:
            roots.sort(key=peak.__getitem__, reverse=True)
            for r in roots[1:]:
                if peak[r] - v < prominence:
                    parent[r] = roots[0]
        owner[p] = roots[0]

    names = {}
    pixels = np.fromiter(owner.keys(), np.int64, len(owner))
    labels = np.fromiter((names.setdefault(find(c), len(names) + 1) for c in owner.values()),
                         np.int32, len(owner))
    out = np.zeros(flat.size, np.int32)
    out[pixels] = labels
    return out.reshape(h + 2, pw)[1:-1, 1:-1].copy()


class HotspotDetector:
    """Finds the regions clearly hotter than the scene and measures each one."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.ambient = None       # scene median, smoothed
        self.threshold = None     # current "hotspot" level

    def detect(self, values, scale):
        cfg = self.cfg
        blurred = cv.GaussianBlur(values, (0, 0), 1.2)
        target = float(np.median(blurred[::2, ::2]))
        self.ambient = target if self.ambient is None else self.ambient + 0.1 * (target - self.ambient)
        self.threshold = self.ambient + max(cfg.hot_delta, cfg.hot_fraction * scale.span)
        prominence = max(cfg.prominence, 0.08 * scale.span)

        h, w = values.shape
        f = max(1, int(cfg.detect_step))
        small = cv.resize(blurred, (w // f, h // f), interpolation=cv.INTER_AREA) if f > 1 else blurred
        small_labels = segment_peaks(small, self.threshold, prominence)
        up = np.repeat(np.repeat(small_labels, f, axis=0), f, axis=1)
        labels = np.zeros((h, w), np.int32)
        labels[:up.shape[0], :up.shape[1]] = up
        labels[blurred <= self.threshold] = 0

        found = []
        for label in range(1, int(small_labels.max()) + 1):
            mask = labels == label
            ys, xs = np.nonzero(mask)
            if xs.size < cfg.min_area:
                continue
            weight = np.clip(blurred[ys, xs] - self.ambient, 1e-3, None)
            if blurred[ys, xs].max() < self.threshold + cfg.min_rise:
                continue                      # barely warm: texture or the edge of something
            readings = values[ys, xs]
            k = int(np.argmax(readings))
            x0, y0 = int(xs.min()), int(ys.min())
            found.append(Detection(
                box=(x0, y0, int(xs.max()) - x0 + 1, int(ys.max()) - y0 + 1),
                center=(float((xs * weight).sum() / weight.sum()),
                        float((ys * weight).sum() / weight.sum())),
                peak=float(readings[k]),
                peak_xy=(int(xs[k]), int(ys[k])),
                mean=float(readings.mean()),
                area=int(xs.size),
                mask=mask,
            ))
        return found


class Track:
    """One hotspot followed over time, with everything recorded about it."""

    def __init__(self, det, now, cfg):
        self.id = None                    # given once the hotspot is confirmed
        self.det = det
        self.center = self.start = self.pos = det.center   # smoothed centre / start / raw centre
        self.vel = (0.0, 0.0)                              # px per second, smoothed
        self.first_seen = self.last_seen = now
        self.hits, self.missed = 0, 0
        self.distance = 0.0
        self.high, self.low, self.total = -np.inf, np.inf, 0.0
        self.log = deque(maxlen=cfg.log_length)   # (t, peak, mean, cx, cy, bx, by, bw, bh)
        self.see(det, now)

    def see(self, det, now):
        if self.hits:
            dt = now - self.last_seen
            if dt > 0:
                vx = (det.center[0] - self.pos[0]) / dt
                vy = (det.center[1] - self.pos[1]) / dt
                self.vel = (self.vel[0] + 0.4 * (vx - self.vel[0]), self.vel[1] + 0.4 * (vy - self.vel[1]))
            self.pos = det.center
            cx, cy = self.center
            nx, ny = cx + 0.5 * (det.center[0] - cx), cy + 0.5 * (det.center[1] - cy)   # smoothed
            self.distance += float(np.hypot(nx - cx, ny - cy))
            self.center = (nx, ny)
        self.det, self.last_seen = det, now
        self.hits += 1
        self.missed = 0
        self.high, self.low = max(self.high, det.peak), min(self.low, det.peak)
        self.total += det.peak
        self.log.append((now, det.peak, det.mean, *self.center, *det.box))

    def predict(self, now):
        """Where the hotspot should be now, from its last position and velocity."""
        dt = min(now - self.last_seen, 1.0)
        return self.pos[0] + self.vel[0] * dt, self.pos[1] + self.vel[1] * dt

    @property
    def average(self):
        return self.total / max(self.hits, 1)

    def speed(self, window=1.0):
        """Pixels per second over the last `window` seconds of sightings."""
        if len(self.log) < 2:
            return 0.0
        last = self.log[-1]
        old = last
        for entry in reversed(self.log):
            old = entry
            if entry[0] <= last[0] - window:
                break
        dt = last[0] - old[0]
        return float(np.hypot(last[3] - old[3], last[4] - old[4]) / dt) if dt > 0 else 0.0

    def recent(self, since):
        """Log entries newer than `since`, oldest first."""
        return list(takewhile(lambda e: e[0] >= since, reversed(self.log)))[::-1]


class Tracker:
    """Keeps hotspot IDs stable from frame to frame by matching nearest centres."""

    def __init__(self, cfg):
        self.cfg = cfg
        self.tracks = []
        self.next_id = 1

    def update(self, detections, now, keep=None):
        cfg = self.cfg
        pairs = []
        for i, t in enumerate(self.tracks):
            px, py = t.predict(now)
            reach = cfg.match_distance + cfg.max_speed * min(now - t.last_seen, 0.75)
            for j, d in enumerate(detections):
                dist = float(np.hypot(px - d.center[0], py - d.center[1]))
                if dist <= reach:
                    pairs.append((dist, i, j))
        pairs.sort()
        used_t, used_d = set(), set()
        for _, i, j in pairs:
            if i in used_t or j in used_d:
                continue
            used_t.add(i)
            used_d.add(j)
            self.tracks[i].see(detections[j], now)

        survivors = []
        for i, t in enumerate(self.tracks):
            if i not in used_t:
                t.missed += 1
                limit = cfg.forget_frames if t.id is not None else 1
                if t.missed > limit and (keep is None or t.id != keep):
                    continue
            if t.id is None and t.hits >= cfg.confirm_frames:
                t.id = self.next_id
                self.next_id += 1
            survivors.append(t)
        survivors += [Track(d, now, cfg) for j, d in enumerate(detections) if j not in used_d]
        self.tracks = survivors

    def get(self, tid):
        if tid is None:
            return None
        return next((t for t in self.tracks if t.id == tid), None)

    def listed(self):
        """Confirmed hotspots, by ID."""
        return sorted((t for t in self.tracks if t.id is not None), key=lambda t: t.id)


# ----------------------------------------------------------------------------------------------
# 3D views
# ----------------------------------------------------------------------------------------------

class History:
    """Rolling rows: hottest value in each block of image columns, averaged over each time step."""

    def __init__(self, width, cfg):
        self.step = max(1, width // cfg.columns)
        self.n = width // self.step
        self.x = np.arange(self.n) * self.step + self.step / 2
        self.rows = deque(maxlen=cfg.rows)
        self.reset()

    def reset(self):
        self.rows.clear()
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def add_frame(self, values):
        blurred = cv.GaussianBlur(values, (0, 0), 1.0)        # suppress single hot pixels
        h = blurred.shape[0]
        blocks = blurred[:, :self.n * self.step].reshape(h, self.n, self.step)
        self.total += blocks.max(axis=(0, 2))                  # hottest pixel per column block
        self.count += 1

    def commit(self):
        if self.count == 0:
            return
        row = self.total / self.count
        if not self.rows:
            self.rows.extend([row] * self.rows.maxlen)
        else:
            self.rows.append(row)
        self.total = np.zeros(self.n, np.float32)
        self.count = 0

    def as_array(self):
        return np.array(self.rows, np.float32)


BG, PANEL, EDGE = "#0c0e13", "#151821", "#2b303b"
TEXT, MUTED = "#e3e6ec", "#8b92a3"
SPOT_COLORS = ("#38d1ff", "#7dff6a", "#ff6bd6", "#f4f4f4", "#b69bff", "#00f5c4", "#6ba4ff", "#d4ff3a")

INFERNO = pg.colormap.get("inferno")
LUT_BYTES = INFERNO.getLookupTable(0.0, 1.0, 256)                                   # for the 2D images
LUT_FLOAT = INFERNO.getLookupTable(0.0, 1.0, 256, alpha=True).astype(np.float32) / 255   # for 3D colours


def spot_color(tid):
    return SPOT_COLORS[(tid - 1) % len(SPOT_COLORS)]


def to_bytes(values, scale):
    """Map readings onto 0..255 of the colour scale (fast path for pyqtgraph images)."""
    k = 255.0 / scale.span
    return np.clip((values - scale.low) * k, 0, 255).astype(np.uint8)


# ----------------------------------------------------------------------------------------------
# 3D views (OpenGL)
# ----------------------------------------------------------------------------------------------

class GLPanel:
    """An OpenGL 3D view: image X across, time going back, value up. Drag to rotate, wheel to zoom."""

    SX, SY, SZ = 256.0, 200.0, 110.0         # size of the drawing box in GL units

    def __init__(self, history, cfg, zlabel):
        self.widget = gl.GLViewWidget()
        self.widget.setBackgroundColor(BG)
        self.widget.setMinimumHeight(200)
        self.xs = (history.x / history.x.max() * self.SX).astype(np.float32)
        self.ys = np.linspace(0, self.SY, cfg.rows).astype(np.float32)

        grid = gl.GLGridItem()
        grid.setSize(self.SX, self.SY)
        grid.setSpacing(self.SX / 8, self.SY / 8)
        grid.translate(self.SX / 2, self.SY / 2, 0)
        grid.setColor((255, 255, 255, 35))
        self.widget.addItem(grid)

        edge = np.array([[0, self.SY, 0], [0, self.SY, self.SZ]], np.float32)
        self.widget.addItem(gl.GLLinePlotItem(pos=edge, color=(1, 1, 1, 0.3), width=1))
        font = QtGui.QFont("Segoe UI", 8)
        color = QtGui.QColor(MUTED)
        for pos, text in (((self.SX / 2, -18, 0), "Image X"),
                          ((self.SX + 14, self.SY / 2, 0), "Time  (newest at back)"),
                          ((-10, self.SY + 6, self.SZ * 1.08), zlabel)):
            self.widget.addItem(gl.GLTextItem(pos=pos, text=text, color=color, font=font))
        self.z_low = gl.GLTextItem(pos=(-10, self.SY + 6, 0), text="", color=color, font=font)
        self.z_high = gl.GLTextItem(pos=(-10, self.SY + 6, self.SZ), text="", color=color, font=font)
        self.widget.addItem(self.z_low)
        self.widget.addItem(self.z_high)
        self.widget.setCameraPosition(pos=QtGui.QVector3D(self.SX / 2, self.SY / 2, self.SZ * 0.3),
                                      distance=360, elevation=34, azimuth=-112)

    def set_range(self, low, high, unit):
        self.z_low.setData(text=f"{low:.1f} {unit}")
        self.z_high.setData(text=f"{high:.1f} {unit}")


class SurfacePanel(GLPanel):
    """Surface of the hottest value per column over time, in real units."""

    def __init__(self, history, cfg, scale):
        super().__init__(history, cfg, f"Signal ({scale.label})")
        self.scale = scale
        self.surf = gl.GLSurfacePlotItem(x=self.xs, y=self.ys, z=np.zeros((self.xs.size, self.ys.size), np.float32),
                                         computeNormals=False, smooth=False)
        self.widget.addItem(self.surf)

    def draw(self, grid):
        s = self.scale
        norm = np.clip((grid - s.low) / s.span, 0, 1).T                   # (columns, rows)
        colors = LUT_FLOAT[(norm * 255).astype(np.uint8)].reshape(-1, 4)   # one colour per vertex
        self.surf.setData(z=norm * self.SZ, colors=colors)
        self.set_range(s.low, s.high, s.label)


class WispPanel(GLPanel):
    """Glowing particles for everything hotter than the scene's normal level."""

    PARTICLES = 2

    def __init__(self, history, cfg, scale):
        super().__init__(history, cfg, f"{scale.label} above normal")
        self.cfg, self.scale, self.history = cfg, scale, history
        self.gx, self.gy = (a.ravel() for a in np.meshgrid(self.xs, self.ys))
        rng = np.random.default_rng(7)
        self.jitter = rng.uniform(-0.5, 0.5, (self.PARTICLES, 3, self.gx.size)).astype(np.float32)
        self.fade = np.linspace(0.15, 1.0, cfg.rows)[:, None]
        self.dx, self.dy = self.xs[1] - self.xs[0], self.ys[1] - self.ys[0]
        self.ambient = self.top = None
        self.cloud = gl.GLScatterPlotItem(pos=np.zeros((1, 3), np.float32), color=(0, 0, 0, 0), size=1, pxMode=True)
        self.cloud.setGLOptions("additive")
        self.widget.addItem(self.cloud)

    def draw(self, grid):
        s, cfg = self.scale, self.cfg
        target = float(np.percentile(grid, cfg.ambient_pct))
        self.ambient = target if self.ambient is None else self.ambient + 0.05 * (target - self.ambient)

        contrast = max(cfg.contrast * s.span, 1e-6)
        excess = np.clip(grid - self.ambient, 0, None)
        strength = np.clip(excess / contrast, 0, 1) * self.fade
        cells = np.flatnonzero(strength.ravel() > cfg.min_strength)

        target_top = max(4 * contrast, float(excess.max()) * 1.1)
        self.top = target_top if self.top is None else self.top + 0.2 * (target_top - self.top)
        self.set_range(0.0, self.top, s.label)

        if not cells.size:
            self.cloud.setData(pos=np.zeros((1, 3), np.float32), color=(0, 0, 0, 0), size=1)
            return

        heat = excess.ravel()[cells]
        power = strength.ravel()[cells]
        jit = self.jitter[:, :, cells]                                        # (P, 3, n)
        pos = np.empty((self.PARTICLES * cells.size, 3), np.float32)
        pos[:, 0] = (self.gx[cells] + jit[:, 0] * self.dx).ravel()
        pos[:, 1] = (self.gy[cells] + jit[:, 1] * self.dy).ravel()
        pz = np.clip(heat + jit[:, 2] * contrast * 0.15, 0, None)
        pos[:, 2] = (pz / self.top * self.SZ).ravel()
        power = np.tile(power, self.PARTICLES)
        temp = self.ambient + np.tile(heat, self.PARTICLES)                   # absolute value -> same colours

        colors = LUT_FLOAT[(np.clip((temp - s.low) / s.span, 0, 1) * 255).astype(np.uint8)]
        colors[:, 3] = np.clip(power, 0, 1) * 0.5                            # additive glow: keep it soft
        self.cloud.setData(pos=pos, color=colors, size=3 + 9 * power)


# ----------------------------------------------------------------------------------------------
# The one window
# ----------------------------------------------------------------------------------------------

STYLE = f"""
QWidget#root {{ background: {BG}; }}
QLabel {{ color: {TEXT}; }}
QLabel[muted="true"] {{ color: {MUTED}; }}
QFrame[panel="true"] {{ background: {PANEL}; border: 1px solid {EDGE}; border-radius: 4px; }}
QListWidget {{ background: {PANEL}; border: 1px solid {EDGE}; color: {TEXT}; outline: 0; }}
QListWidget::item {{ padding: 2px 4px; }}
QPushButton {{ background: {PANEL}; color: {TEXT}; border: 1px solid {EDGE}; padding: 6px 10px; border-radius: 3px; }}
QPushButton:hover {{ background: #232838; }}
"""


class Window(QtWidgets.QWidget):
    def __init__(self):
        super().__init__()
        self.closed = False

    def closeEvent(self, event):
        self.closed = True
        event.accept()


def label(text="", muted=False, size=9, bold=False):
    w = QtWidgets.QLabel(text)
    if muted:
        w.setProperty("muted", True)
    font = w.font()
    font.setPointSizeF(size)
    font.setBold(bold)
    w.setFont(font)
    return w


class ThermalApp:
    """Live map with every hotspot boxed, a tracked-only view, both 3D views and data panels.

    Pick a hotspot by clicking its box on the live map or its row in the list. The live map
    keeps showing everything; the tracked view, the 3D views, the data and the chart then
    follow only the picked hotspot.
    """

    ROWS = 8

    def __init__(self, cfg, scale, history, tracker, detector, shape, simulated):
        pg.setConfigOptions(imageAxisOrder="row-major", antialias=False)
        self.qapp = pg.mkQApp()
        self.cfg, self.scale, self.history = cfg, scale, history
        self.tracker, self.detector = tracker, detector
        self.selected = None
        self.message, self.message_until = "", 0.0
        self.row_ids, self.drawn = [], []
        self.frames, self.fps, self.fps_t0 = 0, 0.0, time.monotonic()
        self.soft_key, self.soft = None, None
        self.last_panels, self.history_dirty = -np.inf, True
        self._text_cache = {}
        m = max(0, int(cfg.focus_margin))
        self.kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (2 * m + 1, 2 * m + 1))
        h, w = shape
        self.width, self.height = w, h

        win = self.window = Window()
        win.setObjectName("root")
        win.setWindowTitle("Thermal hotspot tracker")
        win.setStyleSheet(STYLE)
        win.resize(int(cfg.figure_size[0] * 100), int(cfg.figure_size[1] * 100))
        outer = QtWidgets.QVBoxLayout(win)
        outer.setContentsMargins(12, 8, 12, 10)
        outer.setSpacing(6)
        outer.addWidget(label("Thermal hotspot tracker" + ("   ·   SIMULATION" if simulated else ""), size=14, bold=True))
        outer.addWidget(label("Click a box (or a list row) to track one hotspot   ·   N / P  next / previous"
                              "   ·   A  show all   ·   1-8  pick from list   ·   S  save CSV   ·   Q  quit"
                              "   ·   drag the 3D views to rotate them", muted=True))
        body = QtWidgets.QHBoxLayout()
        body.setSpacing(12)
        outer.addLayout(body, 1)
        left = QtWidgets.QVBoxLayout()
        right = QtWidgets.QVBoxLayout()
        body.addLayout(left, 7)
        body.addLayout(right, 3)

        # 2D views: live map (all hotspots) and tracked-only view, sharing one colour bar
        self.maps = pg.GraphicsLayoutWidget()
        self.maps.setBackground(BG)
        self.live_plot, self.live_img, self.live_peak = self._map_plot()
        self.focus_plot, self.focus_img, self.focus_peak = self._map_plot()
        self.cbar = pg.ColorBarItem(values=(scale.low, scale.high), colorMap=INFERNO, interactive=False,
                                    width=12, label=scale.label)
        self.maps.addItem(self.cbar)
        self.live_boxes, self.focus_boxes = [], []
        self.focus_note = pg.TextItem("", color=TEXT, anchor=(0.5, 0.5), fill=pg.mkBrush(BG),
                                      border=pg.mkPen(EDGE))
        self.focus_note.setPos(w / 2, h / 2)
        self.focus_plot.addItem(self.focus_note)
        self.maps.scene().sigMouseClicked.connect(self._on_map_click)
        left.addWidget(self.maps, 11)

        # 3D views
        self.panels = [SurfacePanel(history, cfg, scale), WispPanel(history, cfg, scale)]
        self.panel_titles = []
        row3d = QtWidgets.QHBoxLayout()
        for panel in self.panels:
            col = QtWidgets.QVBoxLayout()
            col.setSpacing(2)
            title = label(size=10)
            self.panel_titles.append(title)
            col.addWidget(title)
            col.addWidget(panel.widget, 1)
            row3d.addLayout(col)
        left.addLayout(row3d, 10)

        # hotspot list
        self.list_title = label(muted=True, bold=True)
        right.addWidget(self.list_title)
        self.list = QtWidgets.QListWidget()
        self.list.setFocusPolicy(QtCore.Qt.FocusPolicy.NoFocus)
        self.list.setSelectionMode(QtWidgets.QAbstractItemView.SelectionMode.NoSelection)
        self.list.setFont(QtGui.QFont("Consolas", 10))
        self.list.setVerticalScrollBarPolicy(QtCore.Qt.ScrollBarPolicy.ScrollBarAlwaysOff)
        for _ in range(self.ROWS):
            self.list.addItem(QtWidgets.QListWidgetItem(""))
        self.list.itemClicked.connect(lambda item: self._pick_row(self.list.row(item)))
        self.list.setFixedHeight(self.ROWS * 22 + 8)
        right.addWidget(self.list)

        # data panel
        info = QtWidgets.QFrame()
        info.setProperty("panel", True)
        box = QtWidgets.QVBoxLayout(info)
        box.setContentsMargins(10, 8, 10, 8)
        self.info_title = label(bold=True)
        self.info_body = label(size=9.5)
        self.info_body.setTextFormat(QtCore.Qt.TextFormat.RichText)
        self.info_body.setAlignment(QtCore.Qt.AlignmentFlag.AlignTop | QtCore.Qt.AlignmentFlag.AlignLeft)
        box.addWidget(self.info_title)
        box.addWidget(self.info_body, 1)
        right.addWidget(info, 5)

        # temperature chart
        self.chart = pg.PlotWidget(background=PANEL)
        pi = self.chart.getPlotItem()
        pi.showGrid(x=True, y=True, alpha=0.25)
        pi.setMenuEnabled(False)
        pi.hideButtons()
        pi.setMouseEnabled(False, False)
        pi.setLabel("bottom", "seconds ago", color=MUTED)
        pi.setLabel("left", scale.label, color=MUTED)
        pi.setXRange(-cfg.chart_seconds, 0, padding=0)
        for axis in ("left", "bottom"):
            pi.getAxis(axis).setTextPen(MUTED)
            pi.getAxis(axis).setPen(EDGE)
        self.chart_lines = []
        self.mean_line = pi.plot(pen=pg.mkPen(MUTED, width=1, style=QtCore.Qt.PenStyle.DashLine))
        self.peak_line = pg.InfiniteLine(angle=0, pen=pg.mkPen(MUTED, width=1, style=QtCore.Qt.PenStyle.DotLine))
        pi.addItem(self.peak_line)
        self.chart_title = label(muted=True, size=8.5)
        right.addWidget(self.chart_title)
        right.addWidget(self.chart, 4)

        # status line and buttons
        self.message_label = label(size=9)
        self.message_label.setStyleSheet("color: #ffd166;")
        right.addWidget(self.message_label)
        buttons = QtWidgets.QHBoxLayout()
        for text, action in (("◀ Prev", lambda: self.cycle(-1)), ("Next ▶", lambda: self.cycle(1)),
                             ("Show all", lambda: self.select(None)), ("Save CSV", self.save_csv)):
            b = QtWidgets.QPushButton(text)
            b.setFocusPolicy(QtCore.Qt.FocusPolicy.NoFocus)
            b.clicked.connect(lambda _checked=False, a=action: a())
            buttons.addWidget(b)
        right.addLayout(buttons)

        # keys
        self.shortcuts = []
        keys = {"Q": self.close, "Escape": self.close, "A": lambda: self.select(None), "0": lambda: self.select(None),
                "S": self.save_csv}
        for k in ("N", "Right", "Down"):
            keys[k] = lambda: self.cycle(1)
        for k in ("P", "Left", "Up"):
            keys[k] = lambda: self.cycle(-1)
        for i in range(1, self.ROWS + 1):
            keys[str(i)] = lambda i=i: self._pick_row(i - 1)
        for seq, fn in keys.items():
            sc = QShortcut(QtGui.QKeySequence(seq), win)
            sc.setContext(QtCore.Qt.ShortcutContext.ApplicationShortcut)
            sc.activated.connect(fn)
            self.shortcuts.append(sc)

        win.show()
        self.pump()

    # ---- building blocks ----

    def _map_plot(self):
        w, h = self.width, self.height
        p = self.maps.addPlot()
        for axis in ("left", "bottom"):
            p.hideAxis(axis)
        p.setMenuEnabled(False)
        p.hideButtons()
        vb = p.getViewBox()
        vb.setMouseEnabled(False, False)
        vb.invertY(True)
        vb.setAspectLocked(True)
        vb.setRange(xRange=(-0.5, w - 0.5), yRange=(-0.5, h - 0.5), padding=0)
        vb.setBorder(pg.mkPen(EDGE))
        image = pg.ImageItem()
        image.setLookupTable(LUT_BYTES)
        image.setImage(np.zeros((2 * h, 2 * w), np.uint8), autoLevels=False, levels=(0, 255))
        image.setRect(QtCore.QRectF(-0.5, -0.5, w, h))
        p.addItem(image)
        peak = pg.ScatterPlotItem(symbol="+", size=16, pen=pg.mkPen("w", width=1.6), brush=None)
        peak.setZValue(10)
        p.addItem(peak)
        return p, image, peak

    def _box(self, plot, pool, i):
        while len(pool) <= i:
            rect = QtWidgets.QGraphicsRectItem(0, 0, 1, 1)
            rect.setBrush(QtGui.QBrush(QtCore.Qt.BrushStyle.NoBrush))
            rect.setZValue(5)
            plot.addItem(rect)
            text = pg.TextItem("", fill=pg.mkBrush(0, 0, 0, 150))
            font = QtGui.QFont("Segoe UI", 8)
            font.setBold(True)
            text.setFont(font)
            text.setZValue(6)
            plot.addItem(text)
            pool.append([rect, text, None])
        return pool[i]

    def _place_box(self, entry, t, width, dashed, alpha):
        rect, text, _ = entry
        x, y, w, h = t.det.box
        color = spot_color(t.id)
        rect.setRect(x - 0.5, y - 0.5, w, h)
        style = QtCore.Qt.PenStyle.DashLine if dashed else QtCore.Qt.PenStyle.SolidLine
        key = (t.id, width, dashed, alpha)
        if entry[2] != key:
            rect.setPen(pg.mkPen(color, width=width, style=style))
            rect.setOpacity(alpha)
            text.setColor(color)
            text.setOpacity(alpha)
            entry[2] = key
        above = y > 14
        right = x > self.width - 60
        text.setAnchor((1 if right else 0, 1 if above else 0))
        text.setPos(x + w - 0.5 if right else x - 0.5, y - 1.5 if above else y + h + 0.5)
        text.setText(f"#{t.id}  {t.det.peak:.1f} {self.scale.label}")
        rect.setVisible(True)
        text.setVisible(True)

    @staticmethod
    def _hide(pool, start=0):
        for rect, text, _ in pool[start:]:
            rect.setVisible(False)
            text.setVisible(False)

    def _set_text(self, widget, text):
        """setText only when the text changed (saves Qt relayouts)."""
        if self._text_cache.get(id(widget)) != text:
            self._text_cache[id(widget)] = text
            widget.setText(text)

    def pump(self):
        self.qapp.processEvents()

    def close(self):
        self.window.close()

    @property
    def closed(self):
        return self.window.closed

    # ---- selection ----

    def select(self, tid):
        if tid == self.selected:
            return
        self.selected = tid
        self.history.reset()               # 3D views restart with only the picked hotspot
        self.soft_key = None
        self.last_panels = -np.inf         # refresh the list, data and chart right away
        self.flash(f"Tracking hotspot #{tid} only" if tid is not None else "Showing all hotspots")

    def cycle(self, step):
        ids = [t.id for t in self.tracker.listed()]
        if not ids:
            self.flash("No hotspots to pick")
            return
        if self.selected in ids:
            self.select(ids[(ids.index(self.selected) + step) % len(ids)])
        else:
            self.select(ids[0] if step > 0 else ids[-1])

    def _pick_row(self, row):
        if 0 <= row < len(self.row_ids):
            self.select(self.row_ids[row])

    def flash(self, text, seconds=3.0):
        self.message, self.message_until = text, time.monotonic() + seconds

    def save_csv(self):
        t = self.tracker.get(self.selected)
        if t is None:
            self.flash("Pick a hotspot first, then save its data")
            return
        name = f"hotspot_{t.id}_{time.strftime('%Y%m%d_%H%M%S')}.csv"
        data = np.array(t.log, np.float64)
        data[:, 0] -= t.first_seen
        np.savetxt(name, data, delimiter=",", fmt="%.3f", comments="",
                   header="time_s,temp_max,temp_mean,center_x,center_y,box_x,box_y,box_w,box_h")
        print(f"Saved {len(data)} samples of hotspot #{t.id} to {name}")
        self.flash(f"Saved {name}")

    # ---- events ----

    def _on_map_click(self, event):
        if event.button() != QtCore.Qt.MouseButton.LeftButton:
            return
        pos = event.scenePos()
        if not self.live_plot.vb.sceneBoundingRect().contains(pos):
            return
        p = self.live_plot.vb.mapSceneToView(pos)
        x, y, pad = p.x(), p.y(), 3
        hits = [t for t in self.drawn
                if t.det.box[0] - pad <= x <= t.det.box[0] + t.det.box[2] + pad
                and t.det.box[1] - pad <= y <= t.det.box[1] + t.det.box[3] + pad]
        if hits:
            self.select(min(hits, key=lambda t: t.det.area).id)
        else:
            self.flash("No hotspot there - click inside a box")

    # ---- per frame ----

    def tick(self, now):
        self.frames += 1
        if now - self.fps_t0 >= 1.0:
            self.fps = self.frames / (now - self.fps_t0)
            self.frames, self.fps_t0 = 0, now

    def focus_values(self, values):
        """The frame with everything except the picked hotspot flattened to the scene median."""
        if self.selected is None:
            return values
        ambient = self.detector.ambient
        t = self.tracker.get(self.selected)
        if t is None:
            return np.full_like(values, ambient)
        key = (t.id, t.hits)
        if key != self.soft_key:
            grown = cv.dilate(t.det.mask.astype(np.uint8), self.kernel).astype(np.float32)
            self.soft = cv.GaussianBlur(grown, (0, 0), 1.5)
            self.soft_key = key
        return (ambient + (values - ambient) * self.soft).astype(np.float32)

    def draw(self, values, focused, now):
        """Refresh the window for this frame. Returns False once it has been closed."""
        s = self.scale
        size = (2 * self.width, 2 * self.height)
        for image, data in ((self.live_img, values), (self.focus_img, focused)):
            image.setImage(cv.resize(to_bytes(data, s), size, interpolation=cv.INTER_LINEAR),
                           autoLevels=False, levels=(0, 255))
        listed = self.tracker.listed()
        picked = self.tracker.get(self.selected)
        self._draw_maps(listed, picked)
        self._set_text(self.message_label, self.message if time.monotonic() < self.message_until else "")

        if self.history_dirty:
            self._draw_3d()
            self.history_dirty = False
        if now - self.last_panels >= self.cfg.full_update_seconds:
            self.last_panels = now
            self.cbar.setLevels((s.low, s.high))
            self._draw_list(listed, picked, now)
            self._draw_info(listed, picked, now)
            self._draw_chart(listed, picked, now)
        self.pump()
        return not self.closed

    def _draw_maps(self, listed, picked):
        shown = [t for t in listed if t.missed <= 2]
        self.drawn = shown
        for i, t in enumerate(shown):
            entry = self._box(self.live_plot, self.live_boxes, i)
            if picked is None:
                self._place_box(entry, t, 1.4, False, 1.0)
            elif t is picked:
                self._place_box(entry, t, 2.6, False, 1.0)
            else:
                self._place_box(entry, t, 1.0, True, 0.45)
        self._hide(self.live_boxes, len(shown))
        n = len(shown)
        self._set_title(self.live_plot, f"LIVE MAP  ·  {n} hotspot{'s' if n != 1 else ''}  ·  click a box to track it")

        if picked is not None:
            entry = self._box(self.focus_plot, self.focus_boxes, 0)
            self._place_box(entry, picked, 2.6, picked.missed != 0, 1.0)
            for marker in (self.live_peak, self.focus_peak):
                marker.setData([picked.det.peak_xy[0]], [picked.det.peak_xy[1]])
            self.focus_note.setVisible(False)
            title = f"TRACKED ONLY  ·  hotspot #{picked.id}"
        else:
            self._hide(self.focus_boxes)
            for marker in (self.live_peak, self.focus_peak):
                marker.setData([], [])
            gone = self.selected is not None
            note = (f"Hotspot #{self.selected} is gone\npick another one" if gone else
                    "Nothing picked: showing every hotspot\nclick a box on the live map")
            if self._text_cache.get(id(self.focus_note)) != note:
                self._text_cache[id(self.focus_note)] = note
                self.focus_note.setText(note)
            self.focus_note.setVisible(gone or not shown)
            title = "TRACKED ONLY  ·  " + (f"#{self.selected} (gone)" if gone else "all hotspots")
        self._set_title(self.focus_plot, title)

    def _set_title(self, plot, text):
        if self._text_cache.get(id(plot)) != text:
            self._text_cache[id(plot)] = text
            plot.setTitle(text, color=TEXT, size="10pt", justify="left")

    def _draw_list(self, listed, picked, now):
        u = self.scale.label
        rows = listed[:self.ROWS]
        self.row_ids = [t.id for t in rows]
        more = len(listed) - len(rows)
        self._set_text(self.list_title, f"HOTSPOTS  ·  {len(listed)}" + (f"  (+{more} not shown)" if more else "")
                       + ("" if rows else "   ·   none right now"))
        for i in range(self.ROWS):
            item = self.list.item(i)
            if i >= len(rows):
                item.setHidden(True)
                continue
            t = rows[i]
            status = "live" if t.missed == 0 else f"missing {now - t.last_seen:.0f}s"
            color = QtGui.QColor(spot_color(t.id))
            if picked is not None and t is not picked:
                color.setAlphaF(0.45)
            item.setText(f"■ #{t.id:<3d}{t.det.peak:6.1f} {u}   peak {t.high:5.1f}   {status}")
            item.setForeground(color)
            item.setBackground(QtGui.QColor("#22314a") if t is picked else QtGui.QColor(0, 0, 0, 0))
            item.setHidden(False)

    def _draw_info(self, listed, picked, now):
        u, cfg = self.scale.label, self.cfg
        if picked is not None:
            t, d = picked, picked.det
            if t.missed == 0:
                status = "tracking"
            elif t.missed <= cfg.forget_frames:
                status = f"searching... {now - t.last_seen:.1f} s"
            else:
                status = f"lost, last seen {now - t.last_seen:.0f} s ago"
            cx, cy = t.center
            rows = [
                ("Hotspot", f"#{t.id}"),
                ("Status", status),
                ("Temp now", f"{d.peak:.1f} {u}" + ("" if t.missed == 0 else "  (last seen)")),
                ("Region mean", f"{d.mean:.1f} {u}"),
                ("Recorded peak", f"{t.high:.1f} {u}"),
                ("Recorded low", f"{t.low:.1f} {u}"),
                ("Recorded avg", f"{t.average:.1f} {u}"),
                ("Recorded for", f"{t.last_seen - t.first_seen:.1f} s  ({t.hits} samples)"),
                ("Position", f"x {cx:.0f}, y {cy:.0f} px"),
                ("Box size", f"{d.box[2]} x {d.box[3]} px  ({d.area} px)"),
                ("Distance moved", f"{t.distance:.0f} px"),
                ("From start", f"{np.hypot(cx - t.start[0], cy - t.start[1]):.0f} px"),
                ("Speed", f"{t.speed():.1f} px/s"),
            ]
            title, color = f"PICKED HOTSPOT  #{t.id}", spot_color(t.id)
        else:
            live = [t for t in listed if t.missed == 0]
            hottest = max(live, key=lambda t: t.det.peak, default=None)
            s = self.scale
            rows = [
                ("Tracking", "all hotspots" if self.selected is None else f"#{self.selected} (gone)"),
                ("Hotspots now", str(len(live))),
                ("Hottest", f"#{hottest.id}   {hottest.det.peak:.1f} {u}" if hottest else "-"),
                ("Scene median", f"{self.detector.ambient:.1f} {u}"),
                ("Hotspot above", f"{self.detector.threshold:.1f} {u}"),
                ("Colour scale", f"{s.low:.1f} to {s.high:.1f} {u}"),
                ("Frame rate", f"{self.fps:.1f} fps"),
                ("", ""),
                ("Tip", "click a box or a list row"),
                ("", "to track a single hotspot"),
            ]
            title, color = "OVERVIEW", MUTED
        self._set_text(self.info_title, f'<span style="color:{color}">{title}</span>')
        html = "".join(f'<tr><td style="color:{MUTED}; padding-right:14px">{k}</td><td>{v}</td></tr>'
                       for k, v in rows)
        self._set_text(self.info_body, f'<table cellspacing="0" style="line-height:150%">{html}</table>')

    def _draw_chart(self, listed, picked, now):
        cfg, pi = self.cfg, self.chart.getPlotItem()
        since = now - cfg.chart_seconds
        tracks = [picked] if picked is not None else [t for t in listed if t.missed <= cfg.forget_frames][:8]
        while len(self.chart_lines) < len(tracks):
            self.chart_lines.append(pi.plot())
        lo, hi = np.inf, -np.inf
        for line, t in zip(self.chart_lines, tracks):
            data = np.array([(e[0], e[1], e[2]) for e in t.recent(since)]).reshape(-1, 3)
            line.setData(data[:, 0] - now, data[:, 1])
            line.setPen(pg.mkPen(spot_color(t.id), width=2.0 if picked is not None else 1.3))
            line.setVisible(True)
            if len(data):
                lo, hi = min(lo, data[:, 1:].min()), max(hi, data[:, 1].max())
            if t is picked:
                self.mean_line.setData(data[:, 0] - now, data[:, 2])
        for line in self.chart_lines[len(tracks):]:
            line.setVisible(False)
        self.mean_line.setVisible(picked is not None)
        if picked is not None:
            self.peak_line.setValue(picked.high)
            hi = max(hi, picked.high)
        self.peak_line.setVisible(picked is not None)
        if not np.isfinite(lo):
            lo, hi = self.scale.low, self.scale.high
        pad = max(0.5, 0.08 * (hi - lo))
        pi.setYRange(lo - pad, hi + pad, padding=0)
        self._set_text(self.chart_title,
                       f"TEMPERATURE  #{picked.id}   (solid max, dashed mean, dotted recorded peak)"
                       if picked is not None else "TEMPERATURE  ·  all hotspots (max)")

    def _draw_3d(self):
        if not self.history.rows:
            return
        grid = cv.GaussianBlur(self.history.as_array(), (0, 0),
                               sigmaX=self.cfg.blur_x, sigmaY=self.cfg.blur_t)
        for panel in self.panels:
            panel.draw(grid)
        what = f"hotspot #{self.selected}" if self.selected is not None else "all hotspots"
        self._set_text(self.panel_titles[0], f"3D map  ·  {what}")
        self._set_text(self.panel_titles[1], f"Wisps  ·  {what}")


def run(cfg=Config()):
    camera = make_camera(cfg)
    if cfg.simulate:
        print("SIMULATION: synthetic thermal scene (set Config.simulate=False for the real camera).")
    else:
        print("Reading temperatures (°C)." if camera.raw else "Reading brightness only: this camera sends processed frames.")
    detector, tracker = HotspotDetector(cfg), Tracker(cfg)
    app = history = None
    last_update = 0.0

    try:
        while True:
            values = camera.read(idle=app.pump if app is not None else None)
            if values is None or (app is not None and app.closed):
                break
            now = time.monotonic()
            camera.scale.update(values)
            tracker.update(detector.detect(values, camera.scale), now,
                           keep=app.selected if app is not None else None)

            if app is None:
                history = History(values.shape[1], cfg)
                app = ThermalApp(cfg, camera.scale, history, tracker, detector, values.shape, cfg.simulate)

            app.tick(now)
            focused = app.focus_values(values)
            history.add_frame(focused)
            if now - last_update >= cfg.update_seconds:
                last_update = now
                history.commit()
                app.history_dirty = True
            if not app.draw(values, focused, now):
                break
    finally:
        camera.release()
        if app is not None:
            app.window.close()
            app.window.deleteLater()
            app.pump()


if __name__ == "__main__":
    run()